In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.dim_customer AS
SELECT
    customerkey,
    gender,
    continent,
    country,
    state,
    city
FROM electronics_cat.silver.customers;

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.dim_product AS
SELECT
    productkey,
    product_name,
    brand,
    color,
    category,
    subcategory,
    unit_price_usd
FROM electronics_cat.silver.products;

CREATE OR REPLACE TABLE electronics_cat.gold.dim_store AS
SELECT
    store_key,
    country AS store_country,
    state,
    square_meters
FROM electronics_cat.silver.stores;

CREATE OR REPLACE TABLE electronics_cat.gold.dim_date AS
SELECT DISTINCT
    order_date AS date,
    YEAR(order_date) AS year,
    MONTH(order_date) AS month,
    DAY(order_date) AS day
FROM electronics_cat.silver.sales
WHERE order_date IS NOT NULL;

CREATE OR REPLACE TABLE electronics_cat.gold.dim_exchange_rate AS
SELECT
    date,
    currency,
    exchange
FROM electronics_cat.silver.exc_rate;

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.fact_sales AS
SELECT
    s.order_number,
    s.line_item,

    s.order_date,
    d.year,
    d.month,

    s.customerkey,
    s.product_key,
    s.storekey,

    s.quantity,

    p.unit_price_usd,

    er.exchange,

    -------------------------------------------------------------
    ROUND(
        (s.quantity * p.unit_price_usd) / er.exchange,
        2
    ) AS revenue_usd,

    -------------------------------------------------------------
    DATEDIFF(s.delivery_date, s.order_date) AS delivery_days,

    -------------------------------------------------------------
    CASE 
        WHEN s.storekey = -1 THEN 'online'
        ELSE 'store'
    END AS channel,

    s.currency_code

FROM electronics_cat.silver.sales s
LEFT JOIN electronics_cat.gold.dim_product p
    ON s.product_key = p.productkey
LEFT JOIN electronics_cat.gold.dim_date d
    ON s.order_date = d.date
LEFT JOIN electronics_cat.gold.dim_exchange_rate er
    ON s.currency_code = er.currency
    AND s.order_date = er.date;

In [0]:
SELECT
    f.year,
    f.month,
    ROUND(SUM(f.revenue_usd),2) AS revenue_usd
FROM electronics_cat.gold.fact_sales f
WHERE f.year = 2024
GROUP BY f.year, f.month
ORDER BY f.month;

In [0]:
WITH monthly AS (
    SELECT month, SUM(revenue_usd) AS revenue
    FROM electronics_cat.gold.fact_sales
    WHERE year = 2024
    GROUP BY month
),
total AS (
    SELECT SUM(revenue) AS total_rev FROM monthly
)
SELECT
    m.month,
    ROUND(m.revenue,2),
    ROUND(m.revenue * 100 / t.total_rev,2) AS percent
FROM monthly m, total t
ORDER BY m.revenue DESC
LIMIT 3;

In [0]:
SELECT
    p.category,
    ROUND(SUM(f.revenue_usd),2) AS revenue,
    ROUND(
        SUM(f.revenue_usd)*100 / SUM(SUM(f.revenue_usd)) OVER(),2
    ) AS percent
FROM electronics_cat.gold.fact_sales f
JOIN electronics_cat.gold.dim_product p
    ON f.product_key = p.productkey
WHERE f.year = 2024
GROUP BY p.category
ORDER BY revenue DESC
LIMIT 3;

In [0]:
SELECT
    ROUND(AVG(delivery_days),2),
    COUNT(*) 
FROM electronics_cat.gold.fact_sales
WHERE delivery_days IS NOT NULL;

In [0]:
SELECT
    s.store_country,
    ROUND(AVG(f.delivery_days),2),
    COUNT(*),
    PERCENTILE(f.delivery_days,0.5)
FROM electronics_cat.gold.fact_sales f
JOIN electronics_cat.gold.dim_store s
    ON f.storekey = s.store_key
GROUP BY s.store_country
ORDER BY AVG(f.delivery_days) DESC
LIMIT 5;

In [0]:
SELECT
    c.continent,

    ROUND(SUM(CASE WHEN channel='online' THEN revenue_usd END) /
          COUNT(DISTINCT CASE WHEN channel='online' THEN order_number END),2),

    ROUND(SUM(CASE WHEN channel='store' THEN revenue_usd END) /
          COUNT(DISTINCT CASE WHEN channel='store' THEN order_number END),2),

    COUNT(DISTINCT CASE WHEN channel='online' THEN order_number END),
    COUNT(DISTINCT CASE WHEN channel='store' THEN order_number END)

FROM electronics_cat.gold.fact_sales f
JOIN electronics_cat.gold.dim_customer c
    ON f.customerkey = c.customerkey
GROUP BY c.continent;

In [0]:
SELECT
    ROW_NUMBER() OVER(ORDER BY SUM(f.quantity) DESC) AS rank,
    p.category,
    SUM(f.quantity),
    ROUND(SUM(f.quantity)*100 / SUM(SUM(f.quantity)) OVER(),2)
FROM electronics_cat.gold.fact_sales f
JOIN electronics_cat.gold.dim_product p
    ON f.product_key = p.productkey
GROUP BY p.category
LIMIT 5;

In [0]:
SELECT
    ROW_NUMBER() OVER(ORDER BY SUM(f.revenue_usd) DESC) AS rank,
    p.category,
    SUM(f.revenue_usd),
    ROUND(SUM(f.revenue_usd)*100 / SUM(SUM(f.revenue_usd)) OVER(),2)
FROM electronics_cat.gold.fact_sales f
JOIN electronics_cat.gold.dim_product p
    ON f.product_key = p.productkey
GROUP BY p.category
LIMIT 5;

In [0]:
SELECT
    c.continent,
    c.gender,
    COUNT(DISTINCT f.customerkey),
    SUM(f.revenue_usd),
    ROUND(SUM(f.revenue_usd)/COUNT(DISTINCT f.customerkey),2)
FROM electronics_cat.gold.fact_sales f
JOIN electronics_cat.gold.dim_customer c
    ON f.customerkey = c.customerkey
GROUP BY c.continent, c.gender;

In [0]:
WITH cust_orders AS (
    SELECT
        c.continent,
        f.customerkey,
        COUNT(DISTINCT f.order_number) AS orders
    FROM electronics_cat.gold.fact_sales f
    JOIN electronics_cat.gold.dim_customer c
        ON f.customerkey = c.customerkey
    GROUP BY c.continent, f.customerkey
)
SELECT
    continent,
    COUNT(*) AS total_customers,
    SUM(CASE WHEN orders >=2 THEN 1 ELSE 0 END) AS repeat_customers,
    ROUND(
        SUM(CASE WHEN orders >=2 THEN 1 ELSE 0 END)*100/COUNT(*),2
    ) AS repeat_rate
FROM cust_orders
GROUP BY continent;

In [0]:
SELECT
    year,
    month,
    ROUND(SUM(revenue_usd),2) AS revenue_usd
FROM electronics_cat.gold.fact_sales
GROUP BY year, month
ORDER BY year, month;

In [0]:
SELECT
    f.year,
    f.month,
    ROUND(SUM(f.revenue_usd),2) AS revenue_usd
FROM electronics_cat.gold.fact_sales f
WHERE f.year = 2021
GROUP BY f.year, f.month
ORDER BY f.month;

In [0]:
WITH monthly AS (
    SELECT month, SUM(revenue_usd) AS revenue
    FROM electronics_cat.gold.fact_sales
    WHERE year = 2021
    GROUP BY month
),
total AS (
    SELECT SUM(revenue) AS total_rev FROM monthly
)
SELECT
    m.month,
    ROUND(m.revenue,2),
    ROUND(m.revenue * 100 / t.total_rev,2) AS percent
FROM monthly m, total t
ORDER BY m.revenue DESC
LIMIT 3;

In [0]:
SELECT
    p.category,
    ROUND(SUM(f.revenue_usd),2) AS revenue,
    ROUND(
        SUM(f.revenue_usd)*100 / SUM(SUM(f.revenue_usd)) OVER(),2
    ) AS percent
FROM electronics_cat.gold.fact_sales f
JOIN electronics_cat.gold.dim_product p
    ON f.product_key = p.productkey
WHERE f.year = 2021
GROUP BY p.category
ORDER BY revenue DESC
LIMIT 3;